# Encrypted Machine Learning: Sklearn Bridge

This tutorial covers `src/concrete_fhe_toolkit/ml/sklearn_bridge.py`. Training directly under encryption can be slow. Often, the best workflow is to train a model on **clear** data using `scikit-learn`, and then convert it into an `FHEModel` for encrypted inference. This module provides functions to automate that conversion and handles the quantization of weights and thresholds.

## Converting a Linear Model

You can convert an sklearn `LogisticRegression` or `LinearRegression` model using `from_sklearn_linear`. We will use a mock object here that acts like a fitted sklearn LogisticRegression model.

In [ ]:
from concrete_fhe_toolkit.ml.sklearn_bridge import from_sklearn_linear

# 1. Mock a fitted sklearn LogisticRegression model
# In reality, this would be: clf = LogisticRegression().fit(X, y)
class MockSklearnLogisticRegression:
    coef_ = [[0.34, 0.12]]
    intercept_ = [-0.75]
    classes_ = [0, 1]

sklearn_clf = MockSklearnLogisticRegression()

# 2. Convert to an FHEModel
# We use scale=100, which means 0.34 becomes 34, 0.12 becomes 12, -0.75 becomes -75
fhe_model = from_sklearn_linear(sklearn_clf, scale=100)

# 3. Check that the weights were scaled properly
print("FHE Model Weights:", fhe_model.weights)
print("FHE Model Bias:", fhe_model.bias)
assert fhe_model.weights == [34, 12]
assert fhe_model.bias == -75

# 4. Compile and Predict (Features must be quantized with the same scale!)
fhe_model.compile(inputset=[[[0, 0]] * 1], batch_size=1)
pred = fhe_model.predict([4, 1])
assert pred == 1

print("\n✅ Encrypted Sklearn Linear Bridge passed!")